# Ветка 3. Учебный и учебно-тематический план программы повышения квалификации

**Тема главы:** демонстрационный отчет по ветке 3: учебный и учебно-тематический план программы повышения квалификации.

**Программа:** «ИИ-инструменты в преподавательской, научной и повседневной работе: от понимания принципов до практического применения».

**Объем:** 72 академических часа.

Этот ноутбук предназначен для включения в Jupyter Book и показывает, как учебный план может быть представлен как воспроизводимый исследовательский объект: данные хранятся отдельно, таблицы и схемы строятся кодом, а методическая интерпретация фиксируется в markdown-ячейках.

## 1. Постановка задачи

Задача главы — представить учебный и учебно-тематический план не как статичный текстовый документ, а как проверяемую структуру данных. Такой формат важен для проекта по двум причинам.

Во-первых, программа ДПО должна сохранять строгую сумму часов: 72 часа, 8 модулей, фиксированное распределение учебной нагрузки. Во-вторых, каждая тема должна быть связана с компетенциями, формами контроля и объектами будущей образовательной платформы. Если эти связи существуют только в текстовом документе, их трудно проверять, переиспользовать и переносить в инженерную разработку.

В этом ноутбуке учебный план рассматривается как модель, из которой можно автоматически получать таблицы для РПМ, календарный график, карты компетенций, схемы контроля и требования к платформенной реализации.

## 2. Контекст ветки

Ветка 3 не разрабатывает тексты лекций и не заменяет фонд оценочных средств. Ее предмет — учебная организация программы: модули, темы, часы, виды деятельности, календарная логика, компетенции, контроль и связь с платформой.

Результат этой ветки нужен сразу для нескольких последующих работ:

- включение учебного и учебно-тематического плана в РПМ ДПО;
- формирование карты уроков и видеолекций;
- настройка шаблона курса в образовательной платформе;
- проверка полноты компетентностной модели;
- подготовка фонда оценочных средств;
- проектирование календаря запуска курса.

## 3. Исходные структурированные данные

Данные главы вынесены в два файла:

- `data/stage_03_curriculum_plan.yaml` — человекочитаемая версия;
- `data/stage_03_curriculum_plan.json` — машинно-читаемая версия.

Оба файла содержат одну и ту же модель: метаданные главы, методическое основание, структуру курса, модули, темы, календарь, компетенции, требования к платформе, риски и критерии приемки.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd() / 'data'

json_path = DATA_DIR / 'stage_03_curriculum_plan.json'
yaml_path = DATA_DIR / 'stage_03_curriculum_plan.yaml'

with json_path.open('r', encoding='utf-8') as f:
    data = json.load(f)

with yaml_path.open('r', encoding='utf-8') as f:
    data_from_yaml = yaml.safe_load(f)

assert data['chapter']['id'] == data_from_yaml['chapter']['id']
assert data['course']['total_hours'] == 72

print('Данные загружены')
print('Название главы:', data['chapter']['title'])
print('Объем курса:', data['course']['total_hours'], 'часа')
print('Количество модулей:', len(data['course']['modules']))

## 4. Нормализация модели курса

Чтобы работать с учебным планом как с данными, удобно ввести простой объект модуля. Это не объект платформы и не модель базы данных, а исследовательское представление для анализа: номер модуля, название, объем часов, распределение по видам работы и перечень компетенций.

In [ ]:
@dataclass(frozen=True)
class CourseModule:
    module_no: int
    title: str
    hours_total: int
    lecture: int
    practice: int
    self_study: int
    control: int
    competencies: tuple

    @classmethod
    def from_dict(cls, item: dict):
        h = item['hours']
        return cls(
            module_no=item['module_no'],
            title=item['title'],
            hours_total=item['hours_total'],
            lecture=h['lecture'],
            practice=h['practice'],
            self_study=h['self_study'],
            control=h['control'],
            competencies=tuple(item['competencies']),
        )

modules = [CourseModule.from_dict(item) for item in data['course']['modules']]

for m in modules:
    assert m.hours_total == m.lecture + m.practice + m.self_study + m.control

print('Нормализовано модулей:', len(modules))
print('Проверка сумм часов по каждому модулю выполнена')

## 5. Таблица распределения часов по модулям

Следующая таблица является ядром учебного плана. Она фиксирует обязательное распределение 72 часов между 8 модулями и видами учебной деятельности: лекционный блок, практика, самостоятельная работа и контроль.

In [ ]:
module_hours = pd.DataFrame([
    {
        'Модуль': f"{m.module_no}. {m.title}",
        'Всего': m.hours_total,
        'Лекции': m.lecture,
        'Практика': m.practice,
        'Самостоятельная работа': m.self_study,
        'Контроль': m.control,
        'Компетенции': ', '.join(m.competencies),
    }
    for m in modules
])

totals = module_hours[['Всего', 'Лекции', 'Практика', 'Самостоятельная работа', 'Контроль']].sum()
print(totals)
module_hours

## 6. Диаграмма учебной нагрузки

Диаграмма показывает, что программа не является набором лекций. В ней практическая работа занимает больше часов, чем лекционный блок. Это соответствует цели курса: сформировать профессиональное действие преподавателя, а не только передать сведения о технологии.

In [ ]:
plot_df = module_hours.set_index('Модуль')[['Лекции', 'Практика', 'Самостоятельная работа', 'Контроль']]
ax = plot_df.plot(kind='barh', stacked=True, figsize=(12, 7))
ax.set_title('Распределение 72 часов по модулям и видам учебной деятельности')
ax.set_xlabel('Академические часы')
ax.set_ylabel('')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Учебно-тематический план как таблица тем

На уровне РПМ требуется не только распределение часов по модулям, но и детализация тем внутри каждого модуля. Ниже формируется единая таблица тем: номер модуля, тема, часы, формат работы, компетенции и ожидаемый результат.

In [ ]:
topic_rows = []
for module in data['course']['modules']:
    for topic in module['topics']:
        topic_rows.append({
            'Модуль': module['module_no'],
            'Название модуля': module['title'],
            'Тема': topic['topic_no'],
            'Название темы': topic['title'],
            'Всего': topic['hours_total'],
            'Лекции': topic['lecture'],
            'Практика': topic['practice'],
            'Самостоятельная работа': topic['self_study'],
            'Контроль': topic['control'],
            'Формат': topic['format'],
            'Компетенции': ', '.join(topic['competencies']),
            'Результат': topic['result'],
        })

topics_df = pd.DataFrame(topic_rows)

print('Количество тематических блоков:', len(topics_df))
print('Сумма часов по темам:', topics_df['Всего'].sum())
topics_df.head(12)

## 8. Проверка согласованности тематического плана

Тематический план должен быть внутренне непротиворечивым: сумма часов по темам должна совпадать с суммой часов по модулям, а распределение по видам учебной деятельности должно давать 72 часа.

In [ ]:
check_by_module = topics_df.groupby(['Модуль', 'Название модуля'], as_index=False)[['Всего', 'Лекции', 'Практика', 'Самостоятельная работа', 'Контроль']].sum()
check_by_module['Проверка'] = check_by_module.apply(
    lambda row: 'ok' if row['Всего'] == row['Лекции'] + row['Практика'] + row['Самостоятельная работа'] + row['Контроль'] else 'ошибка',
    axis=1,
)

assert int(check_by_module['Всего'].sum()) == data['course']['total_hours']
assert set(check_by_module['Проверка']) == {'ok'}

check_by_module

## 9. Календарный график реализации

Для асинхронной программы удобно поддерживать несколько календарных режимов. В данных зафиксированы два варианта: рекомендованный четырехнедельный и интенсивный двухнедельный. Они не меняют объем программы и структуру модулей, а только распределяют прохождение по времени.

In [ ]:
calendar_df = pd.DataFrame(data['course']['calendar'])
calendar_df

## 10. Визуализация календарного графика

Диаграмма ниже показывает распределение учебной нагрузки по календарным периодам. Ее задача — быстро проверить, не возникла ли перегрузка в одном из периодов и сохраняется ли общий объем 72 часа.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for variant, group in calendar_df.groupby('variant'):
    ax.plot(group['period'], group['hours'], marker='o', label=variant)

ax.set_title('Календарные варианты реализации программы')
ax.set_xlabel('Учебный период')
ax.set_ylabel('Часы')
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

calendar_df.groupby('variant')['hours'].sum()

## 11. Компетентностная карта

Курс формирует десять профессиональных компетенций. На этом этапе важно показать, как компетенции распределены по модулям и где каждая компетенция подтверждается учебными действиями.

In [ ]:
competencies = data['course']['competencies']
competency_rows = []
for code, description in competencies.items():
    related_modules = [
        module['module_no']
        for module in data['course']['modules']
        if code in module['competencies']
    ]
    related_topics = [
        topic['topic_no']
        for module in data['course']['modules']
        for topic in module['topics']
        if code in topic['competencies']
    ]
    competency_rows.append({
        'Компетенция': code,
        'Формулировка': description,
        'Модули': ', '.join(map(str, related_modules)),
        'Количество тем': len(related_topics),
        'Темы': ', '.join(related_topics),
    })

competency_df = pd.DataFrame(competency_rows)
competency_df

## 12. Граф связей модулей и компетенций

Граф показывает, какие модули обеспечивают формирование компетенций. Он полезен для проверки полноты программы: если компетенция не связана ни с одним модулем, она не может быть подтверждена; если модуль не связан с компетенциями, его место в программе требует пересмотра.

In [ ]:
G = nx.Graph()
for module in data['course']['modules']:
    module_node = f"М{module['module_no']}"
    G.add_node(module_node, kind='module', label=module['title'])
    for comp in module['competencies']:
        G.add_node(comp, kind='competency', label=comp)
        G.add_edge(module_node, comp)

pos = nx.spring_layout(G, seed=42, k=0.85)
module_nodes = [n for n, attr in G.nodes(data=True) if attr['kind'] == 'module']
competency_nodes = [n for n, attr in G.nodes(data=True) if attr['kind'] == 'competency']

plt.figure(figsize=(12, 8))
nx.draw_networkx_edges(G, pos, alpha=0.4)
nx.draw_networkx_nodes(G, pos, nodelist=module_nodes, node_size=1400)
nx.draw_networkx_nodes(G, pos, nodelist=competency_nodes, node_size=900)
nx.draw_networkx_labels(G, pos, font_size=9)
plt.title('Граф связей: модули программы и компетенции ПК-ИИ')
plt.axis('off')
plt.tight_layout()
plt.show()

## 13. Связь учебного плана с объектами платформы

Учебный план должен быть переносимым в образовательную платформу. Поэтому каждый модуль и контрольный элемент должен быть сопоставим с объектами платформы: `CourseTemplate`, `CourseModule`, `Lesson`, `CourseResource`, `AssessmentItem`, `CourseRun`, `Enrollment`, `LessonProgress`, `Grade`.

In [ ]:
platform_df = pd.DataFrame(data['course']['platform_requirements'])
platform_df

## 14. Граф перехода от методической структуры к платформенной структуре

Эта схема показывает, как методический учебный план превращается в цифровой курс: от шаблона курса и модулей к урокам, материалам, заданиям, прогрессу, оцениванию и документу об обучении.

In [ ]:
platform_edges = [
    ('Учебный план', 'CourseTemplate'),
    ('CourseTemplate', 'CourseModule'),
    ('CourseModule', 'Lesson'),
    ('Lesson', 'CourseResource'),
    ('CourseModule', 'AssessmentItem'),
    ('CourseTemplate', 'CourseRun'),
    ('CourseRun', 'Enrollment'),
    ('Enrollment', 'LessonProgress'),
    ('Enrollment', 'AssignmentSubmission'),
    ('AssessmentItem', 'AssignmentSubmission'),
    ('AssignmentSubmission', 'SubmissionAttempt'),
    ('SubmissionAttempt', 'AssignmentReview'),
    ('AssignmentReview', 'Grade'),
    ('Grade', 'IssuedCertificate'),
]

DG = nx.DiGraph()
DG.add_edges_from(platform_edges)
pos = nx.spring_layout(DG, seed=7, k=0.95)

plt.figure(figsize=(13, 8))
nx.draw_networkx_edges(DG, pos, arrows=True, arrowstyle='-|>', arrowsize=14, alpha=0.5)
nx.draw_networkx_nodes(DG, pos, node_size=1800)
nx.draw_networkx_labels(DG, pos, font_size=8)
plt.title('Переход от учебного плана к структуре образовательной платформы')
plt.axis('off')
plt.tight_layout()
plt.show()

## 15. Формы контроля по модулям

На уровне ветки 3 контроль фиксируется как элемент учебного плана. Детальная разработка тестовых вопросов и рубрик относится к ветке фонда оценочных средств, однако уже здесь необходимо указать, какой тип контроля завершает каждый модуль.

In [ ]:
control_df = pd.DataFrame([
    {
        'Модуль': module['module_no'],
        'Название': module['title'],
        'Контрольные часы': module['hours']['control'],
        'Форма контроля': module['control_form'],
        'AssessmentItem': module['assessment_item'],
        'Компетенции': ', '.join(module['competencies']),
    }
    for module in data['course']['modules']
])

control_df

## 16. Жизненный цикл прохождения программы

Жизненный цикл показывает, как слушатель проходит путь от назначения курса до итогового статуса. Для ветки 3 важно, что учебный план задает не только темы, но и последовательность фиксируемых учебных событий: открытие уроков, выполнение заданий, контроль, прогресс и завершение.

In [ ]:
lifecycle_edges = [
    ('Назначение курса', 'Открытие модуля'),
    ('Открытие модуля', 'Просмотр материалов'),
    ('Просмотр материалов', 'Практическое задание'),
    ('Практическое задание', 'Отправка артефакта'),
    ('Отправка артефакта', 'Проверка'),
    ('Проверка', 'Доработка'),
    ('Доработка', 'Повторная отправка'),
    ('Повторная отправка', 'Проверка'),
    ('Проверка', 'Зачет модуля'),
    ('Зачет модуля', 'Следующий модуль'),
    ('Следующий модуль', 'Открытие модуля'),
    ('Зачет модуля', 'Итоговый проект'),
    ('Итоговый проект', 'Защита'),
    ('Защита', 'Итоговый статус'),
]

LC = nx.DiGraph()
LC.add_edges_from(lifecycle_edges)
pos = nx.spring_layout(LC, seed=3, k=1.05)

plt.figure(figsize=(13, 8))
nx.draw_networkx_edges(LC, pos, arrows=True, arrowstyle='-|>', arrowsize=14, alpha=0.45)
nx.draw_networkx_nodes(LC, pos, node_size=1800)
nx.draw_networkx_labels(LC, pos, font_size=8)
plt.title('Жизненный цикл прохождения программы слушателем')
plt.axis('off')
plt.tight_layout()
plt.show()

## 17. Демонстрационный алгоритм проверки завершения курса

Ниже приведен упрощенный алгоритм, который показывает, как учебный план может использоваться для проверки прохождения. Это не промышленная логика платформы, а воспроизводимая демонстрация: курс завершен, если все обязательные модульные контрольные элементы приняты.

In [ ]:
def build_demo_progress(modules, accepted_until=5):
    rows = []
    for module in modules:
        status = 'accepted' if module.module_no <= accepted_until else 'not_submitted'
        rows.append({
            'module_no': module.module_no,
            'module_title': module.title,
            'assessment_status': status,
            'hours': module.hours_total,
        })
    return pd.DataFrame(rows)


def course_completion_status(progress_df):
    accepted = progress_df['assessment_status'].eq('accepted')
    completed_hours = int(progress_df.loc[accepted, 'hours'].sum())
    total_hours = int(progress_df['hours'].sum())
    return {
        'completed_modules': int(accepted.sum()),
        'total_modules': int(len(progress_df)),
        'completed_hours': completed_hours,
        'total_hours': total_hours,
        'progress_percent': round(completed_hours / total_hours * 100, 1),
        'course_status': 'completed' if accepted.all() else 'in_progress',
    }

demo_progress = build_demo_progress(modules, accepted_until=5)
status = course_completion_status(demo_progress)
print(status)
demo_progress

## 18. Риски этапа и меры управления

Риски в этой ветке связаны не с технической реализацией как таковой, а с методической целостностью. Главные риски: потеря суммы часов, слабая связь с компетенциями, смешение учебного плана с текстами лекций и невозможность переноса в платформу.

In [ ]:
risks_df = pd.DataFrame(data['course']['risks'])
risks_df['probability'] = [3, 2, 3, 3, 2, 4]
risks_df['impact_score'] = [4, 5, 5, 5, 4, 5]
risks_df['risk_score'] = risks_df['probability'] * risks_df['impact_score']

ax = risks_df.plot.scatter(x='probability', y='impact_score', s=risks_df['risk_score'] * 35, figsize=(8, 6))
for idx, row in risks_df.iterrows():
    ax.text(row['probability'] + 0.03, row['impact_score'] + 0.03, str(idx + 1), fontsize=10)
ax.set_title('Матрица рисков этапа')
ax.set_xlabel('Вероятность, экспертная шкала 1-5')
ax.set_ylabel('Влияние, экспертная шкала 1-5')
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
ax.grid(True)
plt.tight_layout()
plt.show()

risks_df[['risk', 'impact', 'mitigation', 'risk_score']]

## 19. Критерии приемки результата ветки

Критерии приемки нужны для того, чтобы учебно-тематический план можно было считать готовым к включению в РПМ и передаче в разработку платформы.

In [ ]:
acceptance_df = pd.DataFrame({
    '№': range(1, len(data['course']['acceptance_criteria']) + 1),
    'Критерий приемки': data['course']['acceptance_criteria'],
})
acceptance_df

## 20. Экспорт производных таблиц

Ноутбук может использоваться не только для просмотра, но и как генератор производных таблиц. Следующая ячейка сохраняет нормализованные таблицы в каталог `build/`. Это удобно для последующей передачи в инженерную среду или включения в другие главы Jupyter Book.

In [ ]:
BUILD_DIR = Path('build')
BUILD_DIR.mkdir(exist_ok=True)

module_hours.to_csv(BUILD_DIR / 'module_hours.csv', index=False)
topics_df.to_csv(BUILD_DIR / 'thematic_plan.csv', index=False)
calendar_df.to_csv(BUILD_DIR / 'calendar_graph.csv', index=False)
competency_df.to_csv(BUILD_DIR / 'competency_map.csv', index=False)
platform_df.to_csv(BUILD_DIR / 'platform_requirements.csv', index=False)
control_df.to_csv(BUILD_DIR / 'control_forms.csv', index=False)
risks_df.to_csv(BUILD_DIR / 'risks.csv', index=False)
acceptance_df.to_csv(BUILD_DIR / 'acceptance_criteria.csv', index=False)

print('Производные таблицы сохранены в:', BUILD_DIR.resolve())

## 21. Научно-публикационный потенциал

Этот ноутбук может стать основой для нескольких публикационных продуктов.

**Методическая статья.** Материал можно развить в статью о проектировании программы повышения квалификации преподавателей по применению ИИ-инструментов. Основной акцент: переход от общей цифровой грамотности к компетентностной модели, где каждая тема связана с артефактом, тестом и профессиональным действием преподавателя.

**Доклад на методической конференции.** Ноутбук дает готовую структуру доклада: проблема, проектирование 72-часовой программы, карта компетенций, учебно-тематический план, контроль, риски внедрения, связь с образовательной платформой.

**Глава исследовательского журнала проекта.** Формат Jupyter Book позволяет показать не только методический текст, но и воспроизводимые данные: таблицы, графы, проверки сумм часов, календарные варианты и схемы перехода в платформенную модель. Это делает разработку прозрачной и проверяемой.

**Будущая научная работа.** После пилотного запуска к этому ноутбуку можно добавить эмпирические данные: прохождение слушателей, результаты тестов, качество артефактов, частоту доработок, типичные ошибки, динамику цифровой уверенности преподавателей. Тогда глава станет основой для исследования эффективности курса.

## 22. Выводы по ветке 3

Учебный и учебно-тематический план в этой главе представлен как управляемая структура данных. Такой подход позволяет одновременно решать методическую и инженерную задачи: план пригоден для включения в РПМ, но при этом уже сопоставлен с объектами образовательной платформы.

Ключевые результаты ветки:

- сохранена структура 72 часа и 8 модулей;
- зафиксировано распределение часов по видам деятельности;
- каждая тема связана с компетенциями;
- определены формы контроля;
- задан календарный график реализации;
- показан переход от учебного плана к объектам платформы;
- сформированы риски и критерии приемки.

Следующая логическая ветка — карта уроков и микролекций, где тематические блоки будут преобразованы в конкретные уроки, видеоматериалы, конспекты, практические задания и элементы прогресса.

In [ ]:
summary = {
    'modules': len(modules),
    'topics': len(topics_df),
    'total_hours': int(module_hours['Всего'].sum()),
    'lecture_hours': int(module_hours['Лекции'].sum()),
    'practice_hours': int(module_hours['Практика'].sum()),
    'self_study_hours': int(module_hours['Самостоятельная работа'].sum()),
    'control_hours': int(module_hours['Контроль'].sum()),
    'competencies': len(data['course']['competencies']),
    'platform_entities': int(platform_df['entity'].nunique()),
    'acceptance_criteria': len(acceptance_df),
}

assert summary['total_hours'] == 72
assert summary['modules'] == 8
assert summary['competencies'] == 10

summary